# 4.1 · 预测建模基础：标签、特征、防泄露、滚动训练

> 模块 3 我们研究了单个因子。但每个因子的预测力都有限，且因子之间存在相关性。
> **模型（model）的作用是把多个因子合成成一个更准确的预测信号，并捕捉非线性与因子交互**。
> 本节先把"预测问题"的地基打牢：怎么定义标签、怎么组织样本、怎么防止数据泄露、怎么滚动训练。

## 一、把选股建模为一个预测问题

对每个股票 $i$ 在每个交易日 $t$：
- **特征向量**：$X_{i,t} = (f_1, f_2, \dots, f_K)$，即该股票在 $t$ 时刻的 $K$ 个标准化因子；
- **标签**：$y_{i,t} = $ 从 $t$ 到未来某时刻的收益（如次日收益）；
- **模型**：学习一个函数 $f: X \mapsto y$，输出预测值 $\hat{y}_{i,t}$；
- 预测值越大，越看好该股票，组合构建时给越高权重（模块 5）。

> 注意：**标签是"未来"的收益，而特征只能用"当时及过去"的信息**。这是防止前视偏差的根本。

## 二、标签设计

| 标签 | 公式 | 适用 |
|---|---|---|
| 次日收益 | $P_{t+1}/P_t - 1$ | 日频调仓 |
| 5 日收益 | $P_{t+5}/P_t - 1$ | 周频调仓，噪声更小 |
| 多日收益 | $P_{t+h}/P_t - 1$ | 与持有期匹配 |

标签选择决定模型风格：短期标签噪声大、样本多；长期标签更平滑但样本少。**标签持有期应与实际调仓周期一致。**

## 三、构建数据集（model_lab.data）：长表

In [1]:
import sys, numpy as np, pandas as pd
import matplotlib.pyplot as plt
sys.path.insert(0, r'C:\Users\gucao\Desktop\Qlib')
import qlib
from qlib.data import D
import config
from model_lab.data import build_panel, split_by_date, predict_ic
plt.rcParams['font.sans-serif'] = ['Microsoft YaHei']
plt.rcParams['axes.unicode_minus'] = False
qlib.init(provider_uri=str(config.QLIB_DAILY_DIR), joblib_backend='threading')
codes = D.list_instruments(D.instruments('all'), as_list=True, freq='day')
F = D.features(codes, ['$close','$volume','$mkt_cap_ard','$pe_ttm','$pb_lf'], freq='day')
close = F['$close'].unstack('instrument')
vol = F['$volume'].unstack('instrument')
mkt = F['$mkt_cap_ard'].unstack('instrument')
pe = F['$pe_ttm'].unstack('instrument'); pb = F['$pb_lf'].unstack('instrument')

df = build_panel(close, vol, mkt, pe, pb, label_horizon=1)
print('样本长表:', df.shape, '（每行=某股某日，列=因子+标签）')
print('因子数:', len(df.columns)-1)
df.head(3)

[10632:MainThread](2026-10-30 01:14:55,031) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[10632:MainThread](2026-10-30 01:14:56,518) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[10632:MainThread](2026-10-30 01:14:56,519) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': WindowsPath('C:/Users/gucao/Desktop/Qlib/data/qlib_daily')}


样本长表: (567311, 14) （每行=某股某日，列=因子+标签）
因子数: 13


SMB 小市值  EP 盈利收益率  BP 账面市值比  REV5 短期反转  REV20 中期反转  \
datetime   instrument                                                        
2018-01-02 SH600000   -0.871560  0.935780  0.880734        NaN         NaN   
           SH600009   -0.275229  0.256881 -0.110092        NaN         NaN   
           SH600010   -0.449541 -0.743119  0.256881        NaN         NaN   

                       MOM120 长期动量  低波20  低波60  低换手20  低成交额20  低振幅20  负偏度60  \
datetime   instrument                                                         
2018-01-02 SH600000            NaN   NaN   NaN    NaN     NaN    NaN    NaN   
           SH600009            NaN   NaN   NaN    NaN     NaN    NaN    NaN   
           SH600010            NaN   NaN   NaN    NaN     NaN    NaN    NaN   

                       低量比     label  
datetime   instrument                 
2018-01-02 SH600000    NaN -0.004717  
           SH600009    NaN -0.007883  
           SH600010    NaN  0.004000

## 四、按时间切分：训练 / 验证 / 测试（防泄露核心）

**绝对不能随机打乱后切分**（会让未来数据泄漏到训练中）。必须按时间：
- 训练集：学习模型参数；
- 验证集：调超参数；
- 测试集：最终评估，**只在最后用一次**。

In [2]:
train, valid, test = split_by_date(df, '2022-12-31', '2023-12-31')
print('训练集:', train.index.get_level_values('datetime').min().date(),
      '~', train.index.get_level_values('datetime').max().date(), train.shape)
print('验证集:', valid.index.get_level_values('datetime').min().date(),
      '~', valid.index.get_level_values('datetime').max().date(), valid.shape)
print('测试集:', test.index.get_level_values('datetime').min().date(),
      '~', test.index.get_level_values('datetime').max().date(), test.shape)

训练集: 2018-01-02 ~ 2022-12-30 (297302, 14)
验证集: 2023-01-03 ~ 2023-12-29 (66783, 14)
测试集: 2024-01-02 ~ 2026-10-28 (203226, 14)


## 五、滚动训练（Walk-Forward）

实盘中模型不是"训练一次用到底"，而是**定期用最新数据重新训练**：
- 例如每季度/每月，用过去 N 年数据训练，预测下一段；
- 训练窗口随时间向前滚动（扩展窗口或滚动窗口）。

这样模型能适应市场风格变化。下面的模型我们都会用"滚动训练 + 样本外预测"。

## 六、一个最简单的基准：等权因子合成

在进入线性/树/深度模型前，先做一个**零训练的基准**：把所有因子的截面 rank 等权平均，作为预测值。
后面所有模型都要和它比较——如果一个复杂模型连等权基准都跑不赢，就没有意义。

In [3]:
feat_cols = [c for c in df.columns if c != 'label']
# 等权：所有因子（已统一到 [-1,1] 且方向一致）取平均
pred_equal = test[feat_cols].mean(axis=1)
ic = predict_ic(pred_equal, test['label'])
print('等权合成基准 测试集 RankIC: 均值 %.4f  IR %.4f  胜率 %.3f'
      % (ic.mean(), ic.mean()/ic.std(), (ic > 0).mean()))

等权合成基准 测试集 RankIC: 均值 0.0398  IR 0.1467  胜率 0.565


## 七、模型评估：预测 IC

模型预测的核心评价指标就是**预测值与真实未来收益的截面 RankIC**（与模块 3 一致）：
- 每天截面上算预测值与真实收益的 Spearman 相关；
- 再看 IC 均值、IR、胜率。

> 此外还会看预测值分布、与基准的对比、以及最终组合的回测（模块 5、6）。

## 八、小结与面试要点

- 模型 = 学习"因子特征 → 未来收益"的映射，把多因子合成更准的预测。
- 标签用未来收益、持有期与调仓周期匹配；特征只用当时及过去信息。
- **数据必须按时间切分（训练/验证/测试），严禁随机打乱**；实盘用滚动训练。
- 先建立简单基准（等权/IC加权），复杂模型必须跑赢基准。

**面试高频：**
- 你怎么防止模型训练中的数据泄露？（答：标签用未来、按时间切分、特征只用历史、滚动训练）
- 训练/验证/测试怎么划分？（答：严格按时间，测试集只最后用一次）
- 为什么需要滚动训练？（答：市场风格变化，模型需定期更新）
- 标签的持有期怎么定？（答：与实际调仓周期一致）

## 动手练习
1. 把标签改成 5 日收益，重新构建数据集，观察等权基准 IC；
2. 只用 3 个因子做等权，和用全部因子比较；
3. 思考：如果训练集包含测试期的因子值，会发生什么？

---
下一节 **4.2 线性模型：OLS、Ridge、Lasso、截面回归**。